# MOOCCubeX preprocessing and chronological splits

This notebook reuses the completed EDA outputs, converts raw watch segments into model-ready user–video interactions, filters sparse data, creates chronological train/validation/test splits, and builds the filtered video–concept–course graph. All persistent outputs are saved under `/content/drive/MyDrive/DataCon/processed`.

The 3 GB behaviour file is streamed and never loaded fully into memory. DuckDB performs the large filtering and splitting operations out of core.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip -q install ijson pyarrow duckdb tqdm psutil


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, math, os, shutil, gc

import duckdb, ijson, numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
from tqdm.auto import tqdm
import torch

ROOT=Path('/content/drive/MyDrive/DataCon')
RAW=ROOT/'MOOCCubeX'; EDA=ROOT/'EDA_outputs'; OUT=ROOT/'processed'
SPLITS=OUT/'splits'; GRAPH=OUT/'graph'; REPORTS=OUT/'reports'; CHECKPOINTS=OUT/'checkpoints'
for p in [OUT,SPLITS,GRAPH,REPORTS,CHECKPOINTS]: p.mkdir(parents=True,exist_ok=True)

FORCE_REBUILD=False
MIN_USER_INTERACTIONS=5
MIN_VIDEO_USERS=5
MIN_WATCH_SECONDS=5.0
SHORT_VIDEO_MAX_SECONDS=600

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('Outputs:',OUT)


## 1. Validate EDA inputs and load eligible videos

In [ ]:
required=[
 EDA/'tables/video_candidate_eda.parquet', RAW/'relations/user-video.json',
 RAW/'relations/concept-video.txt', RAW/'relations/video_id-ccid.txt',
 RAW/'entities/course.json', RAW/'entities/concept.json'
]
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError(missing)

candidates=pd.read_parquet(EDA/'tables/video_candidate_eda.parquet')
eligible=candidates[
 candidates['eligible_initial'].fillna(False)
 & candidates['duration_seconds'].between(1,SHORT_VIDEO_MAX_SECONDS)
 & candidates['video_id'].notna() & candidates['ccid'].notna()
].drop_duplicates('video_id').copy()

duration_by_video=dict(zip(eligible.video_id.astype(str),eligible.duration_seconds.astype(float)))
ccid_by_video=dict(zip(eligible.video_id.astype(str),eligible.ccid.astype(str)))
eligible_ids=set(duration_by_video)
print('Eligible video IDs:',len(eligible_ids))
display(eligible.head())


## 2. Streaming JSON and watch-interval functions

In [ ]:
def first_byte(path):
 with Path(path).open('rb') as f:
  while True:
   b=f.read(1)
   if not b or not b.isspace(): return b

def stream_json(path):
 path=Path(path); b=first_byte(path)
 if b==b'[':
  with path.open('rb') as f: yield from ijson.items(f,'item')
 elif b==b'{':
  with path.open('r',encoding='utf-8') as f:
   for line in f:
    if line.strip(): yield json.loads(line)
 else: raise ValueError(path)

def merged_seconds(intervals,duration):
 clean=[]
 for a,b in intervals:
  a=max(0.0,min(float(a),duration)); b=max(0.0,min(float(b),duration))
  if b>a and math.isfinite(a) and math.isfinite(b): clean.append((a,b))
 if not clean:return 0.0
 clean.sort(); total=0.; left,right=clean[0]
 for a,b in clean[1:]:
  if a<=right:right=max(right,b)
  else:total+=right-left;left,right=a,b
 return total+right-left


## 3. Build event-level interactions

This cell rescans `user-video.json` once. It clips and merges overlapping watch intervals, computes completion ratio and a bounded engagement weight, and checkpoints the result. Rerunning will reuse the checkpoint.


In [ ]:
RAW_INTERACTIONS=CHECKPOINTS/'eligible_interactions_raw.parquet'
if FORCE_REBUILD or not RAW_INTERACTIONS.exists():
 schema=pa.schema([
  ('user_id',pa.string()),('video_id',pa.string()),('ccid',pa.string()),
  ('timestamp',pa.int64()),('last_timestamp',pa.int64()),('duration_seconds',pa.float32()),
  ('watched_seconds',pa.float32()),('playback_seconds',pa.float32()),
  ('completion_ratio',pa.float32()),('segment_count',pa.int16()),
  ('engagement_weight',pa.float32()),('positive',pa.int8())])
 writer=pq.ParquetWriter(RAW_INTERACTIONS,schema,compression='snappy')
 batch=[]; kept=invalid=0
 try:
  for obj in tqdm(stream_json(RAW/'relations/user-video.json'),desc='Preprocessing users'):
   uid=obj.get('user_id')
   if not uid:continue
   for event in obj.get('seq') or []:
    vid=str(event.get('video_id') or '')
    if vid not in eligible_ids:continue
    dur=duration_by_video[vid]; intervals=[]; playback=0.; timestamps=[]; seg_count=0
    for seg in event.get('segment') or []:
     try:
      a=float(seg['start_point']);b=float(seg['end_point']);speed=float(seg.get('speed') or 1.)
      if b<=a or speed<=0:invalid+=1;continue
      intervals.append((a,b));playback+=(b-a)/speed;seg_count+=1
      if seg.get('local_start_time') is not None:timestamps.append(int(seg['local_start_time']))
     except (KeyError,TypeError,ValueError,OverflowError):invalid+=1
    watched=merged_seconds(intervals,dur)
    if watched<MIN_WATCH_SECONDS or not timestamps:continue
    ratio=min(1.,watched/dur)
    engagement=min(1.,0.85*ratio+0.15*min(1.,seg_count/3.))
    batch.append({'user_id':str(uid),'video_id':vid,'ccid':ccid_by_video[vid],
     'timestamp':min(timestamps),'last_timestamp':max(timestamps),'duration_seconds':dur,
     'watched_seconds':watched,'playback_seconds':playback,'completion_ratio':ratio,
     'segment_count':seg_count,'engagement_weight':engagement,'positive':int(ratio>=.20)})
    kept+=1
    if len(batch)>=100000:
     writer.write_table(pa.Table.from_pylist(batch,schema=schema));batch.clear()
  if batch:writer.write_table(pa.Table.from_pylist(batch,schema=schema))
 finally:writer.close()
 json.dump({'kept_events':kept,'invalid_segments':invalid},open(REPORTS/'streaming_summary.json','w'),indent=2)
else:print('Using checkpoint:',RAW_INTERACTIONS)
print(json.load(open(REPORTS/'streaming_summary.json')))


## 4. Deduplicate and iteratively filter sparse users/videos

In [ ]:
try:
 con.close()
except Exception:
 pass
DB=Path('/content/mooccubex_preprocessing.duckdb')
con=duckdb.connect(str(DB)); con.execute("PRAGMA threads=4"); con.execute("PRAGMA memory_limit='9GB'")
for table in ['interactions','core','core_next','ranked','train_base','valid_base','test_base','eval_users','eval_users_next','train','valid','test']:
 con.execute(f'DROP TABLE IF EXISTS {table}')
con.execute(f"CREATE TABLE interactions AS SELECT * FROM read_parquet('{RAW_INTERACTIONS.as_posix()}') WHERE positive=1")
con.execute('''CREATE OR REPLACE TABLE core AS
 SELECT user_id,video_id,any_value(ccid) AS ccid,min("timestamp") AS "timestamp",max(last_timestamp) AS last_timestamp,
 max(duration_seconds) duration_seconds,least(max(duration_seconds),sum(watched_seconds)) watched_seconds,
 sum(playback_seconds) playback_seconds,
 least(1.0,sum(watched_seconds)/nullif(max(duration_seconds),0)) completion_ratio,
 sum(segment_count) segment_count,max(engagement_weight) engagement_weight,1::TINYINT positive
 FROM interactions GROUP BY user_id,video_id''')
print('Deduplicated:',con.execute('SELECT count(*) FROM core').fetchone()[0])

for iteration in range(1,11):
 before=con.execute('SELECT count(*) FROM core').fetchone()[0]
 con.execute(f'''CREATE OR REPLACE TABLE core_next AS
  SELECT c.* FROM core c
  JOIN (SELECT user_id FROM core GROUP BY user_id HAVING count(*)>={MIN_USER_INTERACTIONS}) u USING(user_id)
  JOIN (SELECT video_id FROM core GROUP BY video_id HAVING count(DISTINCT user_id)>={MIN_VIDEO_USERS}) v USING(video_id)''')
 after=con.execute('SELECT count(*) FROM core_next').fetchone()[0]
 con.execute('DROP TABLE core');con.execute('ALTER TABLE core_next RENAME TO core')
 print(f'Iteration {iteration}: {before:,} -> {after:,}')
 if after==before:break

core_stats=con.execute('SELECT count(*) interactions,count(DISTINCT user_id) users,count(DISTINCT video_id) videos FROM core').df()
display(core_stats)


## 5. Chronological train, validation, and test splits

For every user, the newest interaction becomes test, the second newest becomes validation, and all earlier interactions become training. Validation/test items absent from training are removed to prevent item cold-start from distorting evaluation.


In [ ]:
for table in ['ranked','train_base','valid_base','test_base','eval_users','eval_users_next','train','valid','test']:
 con.execute(f'DROP TABLE IF EXISTS {table}')

con.execute('''CREATE TABLE ranked AS SELECT *,
 row_number() OVER(PARTITION BY user_id ORDER BY "timestamp" DESC,video_id) reverse_rank,
 count(*) OVER(PARTITION BY user_id) user_count FROM core''')
con.execute('CREATE TABLE train_base AS SELECT * EXCLUDE(reverse_rank,user_count) FROM ranked WHERE reverse_rank>2')
con.execute('CREATE TABLE valid_base AS SELECT * EXCLUDE(reverse_rank,user_count) FROM ranked WHERE reverse_rank=2')
con.execute('CREATE TABLE test_base AS SELECT * EXCLUDE(reverse_rank,user_count) FROM ranked WHERE reverse_rank=1')
con.execute('''CREATE TABLE eval_users AS SELECT user_id FROM valid_base INTERSECT SELECT user_id FROM test_base''')

for iteration in range(1,11):
 old_count=con.execute('SELECT count(*) FROM eval_users').fetchone()[0]
 for table in ['train','valid','test','eval_users_next']: con.execute(f'DROP TABLE IF EXISTS {table}')
 con.execute('CREATE TABLE train AS SELECT b.* FROM train_base b JOIN eval_users USING(user_id)')
 con.execute('''CREATE TABLE valid AS SELECT b.* FROM valid_base b JOIN eval_users USING(user_id)
  JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')
 con.execute('''CREATE TABLE test AS SELECT b.* FROM test_base b JOIN eval_users USING(user_id)
  JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')
 con.execute('''CREATE TABLE eval_users_next AS SELECT user_id FROM valid INTERSECT SELECT user_id FROM test''')
 new_count=con.execute('SELECT count(*) FROM eval_users_next').fetchone()[0]
 con.execute('DROP TABLE eval_users');con.execute('ALTER TABLE eval_users_next RENAME TO eval_users')
 print(f'Evaluation cleanup {iteration}: {old_count:,} -> {new_count:,} users')
 if new_count==old_count: break

# Rebuild once using the final stable user set.
for table in ['train','valid','test']: con.execute(f'DROP TABLE IF EXISTS {table}')
con.execute('CREATE TABLE train AS SELECT b.* FROM train_base b JOIN eval_users USING(user_id)')
con.execute('''CREATE TABLE valid AS SELECT b.* FROM valid_base b JOIN eval_users USING(user_id)
 JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')
con.execute('''CREATE TABLE test AS SELECT b.* FROM test_base b JOIN eval_users USING(user_id)
 JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')

for name in ['train','valid','test']:
 path=SPLITS/f'{name}.parquet'
 con.execute(f'''COPY (SELECT * FROM {name} ORDER BY user_id,"timestamp") TO '{path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)''')

split_stats=con.execute('''SELECT 'train' split,count(*) interactions,count(DISTINCT user_id) users,count(DISTINCT video_id) videos FROM train
UNION ALL SELECT 'validation',count(*),count(DISTINCT user_id),count(DISTINCT video_id) FROM valid
UNION ALL SELECT 'test',count(*),count(DISTINCT user_id),count(DISTINCT video_id) FROM test''').df()
split_stats.to_csv(REPORTS/'split_statistics.csv',index=False);display(split_stats)


## 6. Create integer ID mappings

In [ ]:
users=con.execute('SELECT DISTINCT user_id FROM train ORDER BY user_id').df();users['user_idx']=np.arange(len(users),dtype=np.int64)
videos=con.execute('SELECT DISTINCT video_id,any_value(ccid) ccid FROM train GROUP BY video_id ORDER BY video_id').df();videos['video_idx']=np.arange(len(videos),dtype=np.int64)
users.to_parquet(GRAPH/'user_index.parquet',index=False);videos.to_parquet(GRAPH/'video_index.parquet',index=False)
print('Indexed users:',len(users),'videos:',len(videos))


## 7. Build filtered video–concept and course–video graph tables

In [ ]:
train_video_ids=set(videos.video_id);train_ccids=set(videos.ccid.dropna())
concept_edges=[]
with (RAW/'relations/concept-video.txt').open('r',encoding='utf-8',errors='replace') as f:
 for line in tqdm(f,desc='Concept-video edges'):
  p=line.rstrip().split('\t')
  if len(p)<2:continue
  a,b=p[0],p[1]
  concept,video_key=(a,b) if a.startswith('K_') else (b,a)
  if video_key in train_ccids:concept_edges.append((concept,video_key))
concept_edges=pd.DataFrame(concept_edges,columns=['concept_id','ccid']).drop_duplicates()
concepts=sorted(concept_edges.concept_id.unique());concept_index=pd.DataFrame({'concept_id':concepts,'concept_idx':np.arange(len(concepts),dtype=np.int64)})
concept_edges.to_parquet(GRAPH/'concept_video_edges.parquet',index=False);concept_index.to_parquet(GRAPH/'concept_index.parquet',index=False)

video_to_ccid=dict(zip(videos.video_id,videos.ccid));course_rows=[]
for course in tqdm(stream_json(RAW/'entities/course.json'),desc='Course-video graph'):
 for r in course.get('resource') or []:
  vid=str(r.get('resource_id') or '')
  if vid in train_video_ids:course_rows.append((str(course.get('id')),vid,video_to_ccid.get(vid),str(r.get('chapter') or ''),' / '.join(str(x) for x in (r.get('titles') or []) if x)))
course_video=pd.DataFrame(course_rows,columns=['course_id','video_id','ccid','chapter','titles']).drop_duplicates(['course_id','video_id'])
course_video.to_parquet(GRAPH/'course_video_edges.parquet',index=False)
print('Concept-video edges:',len(concept_edges),'Course-video edges:',len(course_video))


## 8. Save filtered video metadata and concept names

In [ ]:
video_metadata=eligible[eligible.video_id.isin(train_video_ids)].copy()
video_metadata.to_parquet(GRAPH/'video_metadata.parquet',index=False)
wanted=set(concepts);concept_rows=[]
for obj in tqdm(stream_json(RAW/'entities/concept.json'),desc='Concept metadata'):
 if obj.get('id') in wanted:concept_rows.append({'concept_id':obj.get('id'),'concept_name':obj.get('name'),'context_count':len(obj.get('context') or [])})
pd.DataFrame(concept_rows).to_parquet(GRAPH/'concept_metadata.parquet',index=False)
print('Video metadata:',len(video_metadata),'Concept metadata:',len(concept_rows))


## 9. Validate chronology and leakage

In [ ]:
checks={
 'train_before_validation':con.execute('''SELECT count(*)=0 FROM (SELECT user_id,max("timestamp") t FROM train GROUP BY user_id) a JOIN valid b USING(user_id) WHERE a.t>b."timestamp"''').fetchone()[0],
 'validation_before_test':con.execute('''SELECT count(*)=0 FROM valid a JOIN test b USING(user_id) WHERE a."timestamp">b."timestamp"''').fetchone()[0],
 'validation_items_in_train':con.execute('SELECT count(*)=0 FROM valid v WHERE NOT EXISTS (SELECT 1 FROM train t WHERE t.video_id=v.video_id)').fetchone()[0],
 'test_items_in_train':con.execute('SELECT count(*)=0 FROM test v WHERE NOT EXISTS (SELECT 1 FROM train t WHERE t.video_id=v.video_id)').fetchone()[0],
 'same_validation_test_users':con.execute('''SELECT count(*)=0 FROM
  ((SELECT user_id FROM valid EXCEPT SELECT user_id FROM test)
   UNION ALL
   (SELECT user_id FROM test EXCEPT SELECT user_id FROM valid))''').fetchone()[0],
}
validation=pd.DataFrame(checks.items(),columns=['check','passed']);validation.to_csv(REPORTS/'preprocessing_validation.csv',index=False);display(validation)
if not validation.passed.all():raise AssertionError('Preprocessing validation failed')


## 10. Save manifest and finish

In [ ]:
manifest={
 'created_at':datetime.now(timezone.utc).isoformat(),'min_user_interactions':MIN_USER_INTERACTIONS,
 'min_video_users':MIN_VIDEO_USERS,'minimum_watch_seconds':MIN_WATCH_SECONDS,
 'maximum_video_seconds':SHORT_VIDEO_MAX_SECONDS,'engagement_formula':'0.85*completion_ratio + 0.15*min(1,segment_count/3)',
 'split_method':'per-user chronological leave-two-out','split_statistics':split_stats.to_dict('records'),
 'concept_video_edges':len(concept_edges),'course_video_edges':len(course_video),
 'files':[str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()]
}
json.dump(manifest,open(REPORTS/'preprocessing_manifest.json','w'),indent=2)
print(json.dumps(manifest,indent=2))
print('Preprocessing complete:',OUT)
con.close()


## Preprocessing complete

Use `splits/train.parquet`, `splits/valid.parquet`, and `splits/test.parquet` for model training and evaluation. The `graph` directory contains the user, video and concept index mappings plus the filtered knowledge-graph edges required for explainable recommendations.
